# H01 & H02 Hypothesis Testing with 7 Dummy Data Scenarios

## Overview

This notebook replicates the exact code from **H01 (Confidence Intervals)** and **H02 (Success Rate Estimation)** hypothesis test modules using 7 diverse dummy data scenarios to demonstrate behavior across different data quality conditions.

### 7 Test Scenarios

1. **Scenario 1 (All Populated)**: 30 runs per fault, all metrics present (realistic complete data)
2. **Scenario 2 (All Missing)**: 30 runs per fault, all metrics None (worst case)
3. **Scenario 3 (70% Missing)**: 30 runs per fault, 70% None / 30% values (sparse data)
4. **Scenario 4 (Low Variance)**: 30 runs per fault, all metrics present but near-identical values (tight distribution)
5. **Scenario 5 (Mixed with Outliers)**: 30 runs per fault, normal distribution + extreme outliers (realistic variance)
6. **Scenario 6 (All Identical)**: 30 runs per fault, all metrics have IDENTICAL values (perfect consistency, zero variance)
7. **Scenario 7 (One Sample Per Category)**: 30 runs per fault, exactly 1 non-null run per category, 29 nulls (minimal sample size test)

### Indicators

- **fault_detected**: 1 when `time_to_detect` ≠ 0/null, else 0 (used as filter for H01 and success indicator for H02)
- **fault_mitigated**: 1 when `time_to_mitigate` ≠ 0/null, else 0 (used as filter for H01 and success indicator for H02)

### Metrics Tested

**H01 (Continuous Metrics)**:
- time_to_detect (TTD)
- time_to_mitigate (TTM)
- reasoning_quality_score
- hallucination_score

**H02 (Rate Metrics)**:
- fault_detection_success_rate (using fault_detected indicator)
- fault_mitigation_success_rate (using fault_mitigated indicator)
- rai_compliance_rate
- security_compliance_rate

---

## Section 1: Import Required Libraries

In [3]:
import sys, os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import trim_mean
import warnings

# Add project root to path
project_root = Path(os.getcwd()).parent.parent.parent
sys.path.insert(0, str(project_root))

# Import certified hypothesis test modules
from hypothesis_framework.scripts.hypothesis_tests.h01_confidence_intervals import run_confidence_interval_test
from hypothesis_framework.scripts.hypothesis_tests.h02_success_rate_estimation import run_success_rate_test

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 6)

print("✓ All libraries imported successfully")
print(f"✓ Project root: {project_root}")

✓ All libraries imported successfully
✓ Project root: c:\Users\shiwkumari\Projects\AgentCert\certifier-main


---

## Section 2: Create 5 Dummy Data Scenarios

Define 5 scenarios with varying data quality characteristics. Each scenario contains 30 runs per fault across 3 categories (application_fault, network_fault, resource_fault) with 2-3 sub-faults per category.

In [4]:
def create_scenario_1_all_populated():
    """Scenario 1: All 30 runs with all fields populated (realistic complete data)"""
    rng = np.random.RandomState(42)
    scenarios = {}
    
    categories = {
        "application_fault": ["pod-delete", "pod-restart"],
        "network_fault": ["pod-dns-error", "pod-network-delay", "pod-packet-loss"],
        "resource_fault": ["pod-cpu-hog", "pod-memory-hog"]
    }
    
    for category, faults in categories.items():
        scenarios[category] = {}
        for fault_name in faults:
            runs = []
            for i in range(30):
                run = {
                    "run_id": f"run-{i:03d}",
                    "fault_name": fault_name,
                    "quantitative": {
                        "time_to_detect": float(rng.uniform(1, 60)),  # 1-60 seconds
                        "time_to_mitigate": float(rng.uniform(5, 120)),  # 5-120 seconds
                        "reasoning_quality_score": float(rng.uniform(0.5, 1.0)),  # 0.5-1.0
                        "hallucination_score": float(rng.uniform(0.0, 0.3)),  # 0.0-0.3
                    },
                    "qualitative": {
                        "rai_check_status": rng.choice(["Passed", "Failed"]),
                        "security_compliance_status": rng.choice(["Compliant", "Non-Compliant"]),
                    }
                }
                runs.append(run)
            scenarios[category][fault_name] = runs
    
    return scenarios

def create_scenario_2_all_missing():
    """Scenario 2: All 30 runs with all fields missing (worst case)"""
    scenarios = {}
    
    categories = {
        "application_fault": ["pod-delete", "pod-restart"],
        "network_fault": ["pod-dns-error", "pod-network-delay", "pod-packet-loss"],
        "resource_fault": ["pod-cpu-hog", "pod-memory-hog"]
    }
    
    for category, faults in categories.items():
        scenarios[category] = {}
        for fault_name in faults:
            runs = []
            for i in range(30):
                run = {
                    "run_id": f"run-{i:03d}",
                    "fault_name": fault_name,
                    "quantitative": {
                        "time_to_detect": None,
                        "time_to_mitigate": None,
                        "reasoning_quality_score": None,
                        "hallucination_score": None,
                    },
                    "qualitative": {
                        "rai_check_status": None,
                        "security_compliance_status": None,
                    }
                }
                runs.append(run)
            scenarios[category][fault_name] = runs
    
    return scenarios

def create_scenario_3_70_percent_missing():
    """Scenario 3: 70% missing data (30 runs per fault, 30% populated)"""
    rng = np.random.RandomState(42)
    scenarios = {}
    
    categories = {
        "application_fault": ["pod-delete", "pod-restart"],
        "network_fault": ["pod-dns-error", "pod-network-delay", "pod-packet-loss"],
        "resource_fault": ["pod-cpu-hog", "pod-memory-hog"]
    }
    
    for category, faults in categories.items():
        scenarios[category] = {}
        for fault_name in faults:
            runs = []
            # 30% have data, 70% are missing
            for i in range(30):
                run = {
                    "run_id": f"run-{i:03d}",
                    "fault_name": fault_name,
                    "quantitative": {},
                    "qualitative": {}
                }
                
                if rng.random() < 0.3:  # 30% have data
                    run["quantitative"] = {
                        "time_to_detect": float(rng.uniform(1, 60)),
                        "time_to_mitigate": float(rng.uniform(5, 120)),
                        "reasoning_quality_score": float(rng.uniform(0.5, 1.0)),
                        "hallucination_score": float(rng.uniform(0.0, 0.3)),
                    }
                    run["qualitative"] = {
                        "rai_check_status": rng.choice(["Passed", "Failed"]),
                        "security_compliance_status": rng.choice(["Compliant", "Non-Compliant"]),
                    }
                else:  # 70% missing
                    run["quantitative"] = {
                        "time_to_detect": None,
                        "time_to_mitigate": None,
                        "reasoning_quality_score": None,
                        "hallucination_score": None,
                    }
                    run["qualitative"] = {
                        "rai_check_status": None,
                        "security_compliance_status": None,
                    }
                
                runs.append(run)
            scenarios[category][fault_name] = runs
    
    return scenarios

def create_scenario_4_low_variance():
    """Scenario 4: Low variance (all metrics present, tight distributions)"""
    rng = np.random.RandomState(42)
    scenarios = {}
    
    categories = {
        "application_fault": ["pod-delete", "pod-restart"],
        "network_fault": ["pod-dns-error", "pod-network-delay", "pod-packet-loss"],
        "resource_fault": ["pod-cpu-hog", "pod-memory-hog"]
    }
    
    for category, faults in categories.items():
        scenarios[category] = {}
        for fault_name in faults:
            runs = []
            for i in range(30):
                # Tight distribution: values are nearly identical with small perturbation
                run = {
                    "run_id": f"run-{i:03d}",
                    "fault_name": fault_name,
                    "quantitative": {
                        "time_to_detect": float(5.0 + rng.normal(0, 0.1)),  # ~5.0 ± 0.1
                        "time_to_mitigate": float(10.0 + rng.normal(0, 0.2)),  # ~10.0 ± 0.2
                        "reasoning_quality_score": float(0.85 + rng.normal(0, 0.01)),  # ~0.85 ± 0.01
                        "hallucination_score": float(0.15 + rng.normal(0, 0.01)),  # ~0.15 ± 0.01
                    },
                    "qualitative": {
                        "rai_check_status": "Passed",  # All pass
                        "security_compliance_status": "Compliant",  # All compliant
                    }
                }
                runs.append(run)
            scenarios[category][fault_name] = runs
    
    return scenarios

def create_scenario_5_mixed_with_outliers():
    """Scenario 5: Realistic mixed data with normal distribution + extreme outliers"""
    rng = np.random.RandomState(42)
    scenarios = {}
    
    categories = {
        "application_fault": ["pod-delete", "pod-restart"],
        "network_fault": ["pod-dns-error", "pod-network-delay", "pod-packet-loss"],
        "resource_fault": ["pod-cpu-hog", "pod-memory-hog"]
    }
    
    for category, faults in categories.items():
        scenarios[category] = {}
        for fault_name in faults:
            runs = []
            for i in range(30):
                # 80% normal, 20% extreme outliers
                is_outlier = rng.random() < 0.2
                
                if is_outlier:
                    ttd = float(rng.uniform(200, 300))  # Extreme high TTD
                    ttm = float(rng.uniform(400, 500))  # Extreme high TTM
                    reasoning = float(rng.uniform(0.1, 0.3))  # Low reasoning
                    hallucination = float(rng.uniform(0.7, 0.95))  # High hallucination
                else:
                    ttd = float(rng.normal(25, 10))  # Normal: 25 ± 10
                    ttm = float(rng.normal(50, 20))  # Normal: 50 ± 20
                    reasoning = float(rng.normal(0.8, 0.1))  # Normal: 0.8 ± 0.1
                    hallucination = float(rng.normal(0.15, 0.1))  # Normal: 0.15 ± 0.1
                
                run = {
                    "run_id": f"run-{i:03d}",
                    "fault_name": fault_name,
                    "quantitative": {
                        "time_to_detect": max(0, float(ttd)),
                        "time_to_mitigate": max(0, float(ttm)),
                        "reasoning_quality_score": float(np.clip(reasoning, 0, 1)),
                        "hallucination_score": float(np.clip(hallucination, 0, 1)),
                    },
                    "qualitative": {
                        "rai_check_status": "Failed" if is_outlier else rng.choice(["Passed", "Failed"]),
                        "security_compliance_status": "Non-Compliant" if is_outlier else rng.choice(["Compliant", "Non-Compliant"]),
                    }
                }
                runs.append(run)
            scenarios[category][fault_name] = runs
    
    return scenarios

def create_scenario_6_all_identical():
    """Scenario 6: All 30 runs with identical metric values (perfect consistency)"""
    scenarios = {}
    
    categories = {
        "application_fault": ["pod-delete", "pod-restart"],
        "network_fault": ["pod-dns-error", "pod-network-delay", "pod-packet-loss"],
        "resource_fault": ["pod-cpu-hog", "pod-memory-hog"]
    }
    
    for category, faults in categories.items():
        scenarios[category] = {}
        for fault_name in faults:
            runs = []
            for i in range(30):
                # All runs have IDENTICAL metric values
                run = {
                    "run_id": f"run-{i:03d}",
                    "fault_name": fault_name,
                    "quantitative": {
                        "time_to_detect": 15.0,  # Exactly 15.0s for all runs
                        "time_to_mitigate": 45.0,  # Exactly 45.0s for all runs
                        "reasoning_quality_score": 0.92,  # Exactly 0.92 for all runs
                        "hallucination_score": 0.08,  # Exactly 0.08 for all runs
                    },
                    "qualitative": {
                        "rai_check_status": "Passed",  # All pass
                        "security_compliance_status": "Compliant",  # All compliant
                    }
                }
                runs.append(run)
            scenarios[category][fault_name] = runs
    
    return scenarios

def create_scenario_7_one_sample_per_fault():
    """Scenario 7: Minimal sample size (exactly 1 non-null run per sub-fault, 29 nulls)"""
    scenarios = {}
    
    categories = {
        "application_fault": ["pod-delete", "pod-restart"],
        "network_fault": ["pod-dns-error", "pod-network-delay", "pod-packet-loss"],
        "resource_fault": ["pod-cpu-hog", "pod-memory-hog"]
    }
    
    for category, faults in categories.items():
        scenarios[category] = {}
        for fault_name in faults:
            runs = []
            for i in range(30):
                run = {
                    "run_id": f"run-{i:03d}",
                    "fault_name": fault_name,
                    "quantitative": {},
                    "qualitative": {}
                }
                
                # First run has data, remaining 29 are null
                if i == 0:
                    run["quantitative"] = {
                        "time_to_detect": 22.5,
                        "time_to_mitigate": 55.0,
                        "reasoning_quality_score": 0.88,
                        "hallucination_score": 0.12,
                    }
                    run["qualitative"] = {
                        "rai_check_status": "Passed",
                        "security_compliance_status": "Compliant",
                    }
                else:  # 29 runs with all nulls
                    run["quantitative"] = {
                        "time_to_detect": None,
                        "time_to_mitigate": None,
                        "reasoning_quality_score": None,
                        "hallucination_score": None,
                    }
                    run["qualitative"] = {
                        "rai_check_status": None,
                        "security_compliance_status": None,
                    }
                
                runs.append(run)
            scenarios[category][fault_name] = runs
    
    return scenarios

# Generate all 7 scenarios
print("=" * 90)
print("GENERATING 7 DUMMY DATA SCENARIOS")
print("=" * 90 + "\n")

scenarios_dict = {
    "Scenario 1: All Populated": create_scenario_1_all_populated(),
    "Scenario 2: All Missing": create_scenario_2_all_missing(),
    "Scenario 3: 70% Missing": create_scenario_3_70_percent_missing(),
    "Scenario 4: Low Variance": create_scenario_4_low_variance(),
    "Scenario 5: Mixed with Outliers": create_scenario_5_mixed_with_outliers(),
    "Scenario 6: All Identical": create_scenario_6_all_identical(),
    "Scenario 7: One Sample Per Fault": create_scenario_7_one_sample_per_fault(),
}

for scenario_name, scenario_data in scenarios_dict.items():
    print(f"\n{scenario_name}:")
    total_runs = 0
    for category, faults in scenario_data.items():
        for fault_name, runs in faults.items():
            total_runs += len(runs)
    
    print(f"  Total runs: {total_runs}")
    print(f"  Categories: {list(scenario_data.keys())}")
    print(f"  Total sub-faults: {sum(len(faults) for faults in scenario_data.values())}")

print("\n✓ All scenarios created successfully")

GENERATING 7 DUMMY DATA SCENARIOS


Scenario 1: All Populated:
  Total runs: 210
  Categories: ['application_fault', 'network_fault', 'resource_fault']
  Total sub-faults: 7

Scenario 2: All Missing:
  Total runs: 210
  Categories: ['application_fault', 'network_fault', 'resource_fault']
  Total sub-faults: 7

Scenario 3: 70% Missing:
  Total runs: 210
  Categories: ['application_fault', 'network_fault', 'resource_fault']
  Total sub-faults: 7

Scenario 4: Low Variance:
  Total runs: 210
  Categories: ['application_fault', 'network_fault', 'resource_fault']
  Total sub-faults: 7

Scenario 5: Mixed with Outliers:
  Total runs: 210
  Categories: ['application_fault', 'network_fault', 'resource_fault']
  Total sub-faults: 7

Scenario 6: All Identical:
  Total runs: 210
  Categories: ['application_fault', 'network_fault', 'resource_fault']
  Total sub-faults: 7

Scenario 7: One Sample Per Fault:
  Total runs: 210
  Categories: ['application_fault', 'network_fault', 'resource_fault']
  Tota

In [5]:
def add_indicators_to_scenarios(scenario_data):
    """
    Add fault_detected and fault_mitigated indicators to each run.
    
    - fault_detected = 1 when time_to_detect is not None and not 0, else 0
    - fault_mitigated = 1 when time_to_mitigate is not None and not 0, else 0
    """
    for category, faults in scenario_data.items():
        for fault_name, runs in faults.items():
            for run in runs:
                quantitative = run["quantitative"]
                
                ttd = quantitative.get("time_to_detect")
                ttm = quantitative.get("time_to_mitigate")
                
                # Create indicators: 1 if value exists and > 0, else 0
                run["quantitative"]["fault_detected"] = 1 if (ttd is not None and ttd > 0) else 0
                run["quantitative"]["fault_mitigated"] = 1 if (ttm is not None and ttm > 0) else 0
    
    return scenario_data

# Add indicators to all scenarios
print("\n" + "=" * 90)
print("ADDING FAULT INDICATORS (fault_detected, fault_mitigated)")
print("=" * 90 + "\n")

for scenario_name in scenarios_dict.keys():
    scenarios_dict[scenario_name] = add_indicators_to_scenarios(scenarios_dict[scenario_name])
    
    # Show summary of indicators
    scenario_data = scenarios_dict[scenario_name]
    total_detected = 0
    total_mitigated = 0
    total_runs = 0
    
    for category, faults in scenario_data.items():
        for fault_name, runs in faults.items():
            for run in runs:
                total_runs += 1
                total_detected += run["quantitative"]["fault_detected"]
                total_mitigated += run["quantitative"]["fault_mitigated"]
    
    print(f"\n{scenario_name}:")
    print(f"  Total runs: {total_runs}")
    print(f"  fault_detected = 1: {total_detected} ({total_detected/total_runs*100:.1f}%)")
    print(f"  fault_mitigated = 1: {total_mitigated} ({total_mitigated/total_runs*100:.1f}%)")

print("\n✓ Indicators added successfully")


ADDING FAULT INDICATORS (fault_detected, fault_mitigated)


Scenario 1: All Populated:
  Total runs: 210
  fault_detected = 1: 210 (100.0%)
  fault_mitigated = 1: 210 (100.0%)

Scenario 2: All Missing:
  Total runs: 210
  fault_detected = 1: 0 (0.0%)
  fault_mitigated = 1: 0 (0.0%)

Scenario 3: 70% Missing:
  Total runs: 210
  fault_detected = 1: 63 (30.0%)
  fault_mitigated = 1: 63 (30.0%)

Scenario 4: Low Variance:
  Total runs: 210
  fault_detected = 1: 210 (100.0%)
  fault_mitigated = 1: 210 (100.0%)

Scenario 5: Mixed with Outliers:
  Total runs: 210
  fault_detected = 1: 209 (99.5%)
  fault_mitigated = 1: 210 (100.0%)

Scenario 6: All Identical:
  Total runs: 210
  fault_detected = 1: 210 (100.0%)
  fault_mitigated = 1: 210 (100.0%)

Scenario 7: One Sample Per Fault:
  Total runs: 210
  fault_detected = 1: 7 (3.3%)
  fault_mitigated = 1: 7 (3.3%)

✓ Indicators added successfully


In [6]:
print("\n" + "=" * 100)
print("SAMPLE DATA: TOP 10 ROWS FROM EACH SCENARIO")
print("=" * 100 + "\n")

for scenario_name, scenario_data in scenarios_dict.items():
    print(f"\n{'='*100}")
    print(f"{scenario_name.upper()}")
    print(f"{'='*100}")
    
    # Get first fault from first category
    first_category = list(scenario_data.keys())[0]
    first_fault = list(scenario_data[first_category].keys())[0]
    first_runs = scenario_data[first_category][first_fault][:10]  # Top 10 rows
    
    # Create a DataFrame for display
    rows_list = []
    for run in first_runs:
        row = {
            "run_id": run["run_id"],
            "fault_name": run["fault_name"],
            "ttd": run["quantitative"].get("time_to_detect"),
            "ttm": run["quantitative"].get("time_to_mitigate"),
            "reasoning": run["quantitative"].get("reasoning_quality_score"),
            "hallucination": run["quantitative"].get("hallucination_score"),
            "fault_detected": run["quantitative"].get("fault_detected"),
            "fault_mitigated": run["quantitative"].get("fault_mitigated"),
            "rai_status": run["qualitative"].get("rai_check_status"),
            "security_status": run["qualitative"].get("security_compliance_status"),
        }
        rows_list.append(row)
    
    df = pd.DataFrame(rows_list)
    
    print(f"\nCategory: {first_category}")
    print(f"Fault: {first_fault}")
    print(f"Showing 10 rows (out of 30 per fault):\n")
    print(df.to_string(index=False))
    print()

print("\n✓ Sample data display complete")



SAMPLE DATA: TOP 10 ROWS FROM EACH SCENARIO


SCENARIO 1: ALL POPULATED

Category: application_fault
Fault: pod-delete
Showing 10 rows (out of 30 per fault):

 run_id fault_name       ttd        ttm  reasoning  hallucination  fault_detected  fault_mitigated rai_status security_status
run-000 pod-delete 23.097867 114.332145   0.865997       0.179598               1                1     Passed   Non-Compliant
run-001 pod-delete 10.203677  11.679615   0.933088       0.180335               1                1     Failed       Compliant
run-002 pod-delete  2.214485 116.539633   0.916221       0.063702               1                1     Failed   Non-Compliant
run-003 pod-delete 11.820866  39.987858   0.762378       0.129584               1                1     Passed       Compliant
run-004 pod-delete 37.099321  21.041794   0.646072       0.109909               1                1     Failed   Non-Compliant
run-005 pod-delete 47.325382  27.962485   0.757117       0.177724               1   

---

## Section 3: Run H01 (Confidence Intervals) on All Scenarios

**Hypothesis 1**: Bootstrap BCa CI on IQM (25% trimmed mean) provides reliable bounds on true typical performance.

**Filter Logic**: 
- TTD: only runs where `fault_detected == 1` (using the indicator we created)
- TTM: only runs where `fault_mitigated == 1` (using the indicator we created)

**Skip Condition**: Skip if fewer than 4 samples per category after filtering.

Run all metrics and scenarios, showing skip/execute decisions.

In [7]:
def build_h01_data_for_scenario(scenario_data, metric_field, filter_field=None, filter_value=None):
    """
    Build data for H01 testing with optional filtering.
    
    Implements the same logic as hypothesis_framework.scripts.utils.build_subfault_data()
    
    Args:
        scenario_data: {category: {fault_name: [runs]}}
        metric_field: Field name to extract (e.g., 'time_to_detect', 'hallucination_score')
        filter_field: Optional filter field (e.g., 'fault_detected', 'fault_mitigated')
        filter_value: Expected value for filter (e.g., 1)
    
    Returns: {category: {fault_name: [values]}}
    """
    result = {}
    
    for category, faults in scenario_data.items():
        result[category] = {}
        
        for fault_name, runs in faults.items():
            values = []
            
            for run in runs:
                quantitative = run.get("quantitative", {})
                
                # Apply filter if specified
                if filter_field is not None:
                    if quantitative.get(filter_field) != filter_value:
                        continue
                
                # Extract metric value
                val = quantitative.get(metric_field)
                if val is not None:
                    values.append(val)
            
            if values:
                result[category][fault_name] = values
    
    return result

# Test H01 on all scenarios
print("=" * 90)
print("H01: CONFIDENCE INTERVALS TEST — ALL SCENARIOS")
print("=" * 90 + "\n")

h01_results = {
    "time_to_detect": {},
    "time_to_mitigate": {},
    "reasoning_quality_score": {},
    "hallucination_score": {},
}

for scenario_name, scenario_data in scenarios_dict.items():
    print(f"\n{'='*90}")
    print(f"{scenario_name.upper()}")
    print(f"{'='*90}")
    
    h01_results["time_to_detect"][scenario_name] = {}
    h01_results["time_to_mitigate"][scenario_name] = {}
    h01_results["reasoning_quality_score"][scenario_name] = {}
    h01_results["hallucination_score"][scenario_name] = {}
    
    # ========== H01: time_to_detect (with fault_detected filter) ==========
    print("\n[H01] time_to_detect (filtered by fault_detected == 1):")
    print("-" * 90)
    
    ttd_data = build_h01_data_for_scenario(scenario_data, "time_to_detect", "fault_detected", 1)
    
    # Count eligible samples
    total_samples = sum(len(v) for v in ttd_data.values() for v in v.values() if isinstance(v, list))
    print(f"  Eligible samples (after filtering): {total_samples}")
    
    if total_samples > 0:
        try:
            h01_ttd = run_confidence_interval_test(ttd_data, metric_name="time_to_detect", n_resamples=10000, random_state=42)
            h01_results["time_to_detect"][scenario_name] = h01_ttd
            print(f"  Status: ✓ EXECUTED")
            for c in h01_ttd.per_category:
                if c.n >= 4:
                    print(f"    {c.category:25s}: IQM={c.iqm:7.1f}s [{c.ci_lower:7.1f}, {c.ci_upper:7.1f}]  n={c.n}")
                else:
                    print(f"    {c.category:25s}: ✗ SKIPPED (n={c.n} < 4)")
        except Exception as e:
            print(f"  Status: ✗ ERROR — {str(e)[:60]}")
    else:
        print(f"  Status: ✗ SKIPPED (no eligible samples)")
    
    # ========== H01: time_to_mitigate (with fault_mitigated filter) ==========
    print("\n[H01] time_to_mitigate (filtered by fault_mitigated == 1):")
    print("-" * 90)
    
    ttm_data = build_h01_data_for_scenario(scenario_data, "time_to_mitigate", "fault_mitigated", 1)
    total_samples = sum(len(v) for v in ttm_data.values() for v in v.values() if isinstance(v, list))
    print(f"  Eligible samples (after filtering): {total_samples}")
    
    if total_samples > 0:
        try:
            h01_ttm = run_confidence_interval_test(ttm_data, metric_name="time_to_mitigate", n_resamples=10000, random_state=42)
            h01_results["time_to_mitigate"][scenario_name] = h01_ttm
            print(f"  Status: ✓ EXECUTED")
            for c in h01_ttm.per_category:
                if c.n >= 4:
                    print(f"    {c.category:25s}: IQM={c.iqm:7.1f}s [{c.ci_lower:7.1f}, {c.ci_upper:7.1f}]  n={c.n}")
                else:
                    print(f"    {c.category:25s}: ✗ SKIPPED (n={c.n} < 4)")
        except Exception as e:
            print(f"  Status: ✗ ERROR — {str(e)[:60]}")
    else:
        print(f"  Status: ✗ SKIPPED (no eligible samples)")
    
    # ========== H01: reasoning_quality_score (no filter) ==========
    print("\n[H01] reasoning_quality_score (no filter):")
    print("-" * 90)
    
    reasoning_data = build_h01_data_for_scenario(scenario_data, "reasoning_quality_score")
    total_samples = sum(len(v) for v in reasoning_data.values() for v in v.values() if isinstance(v, list))
    print(f"  Available samples: {total_samples}")
    
    if total_samples > 0:
        try:
            h01_reasoning = run_confidence_interval_test(reasoning_data, metric_name="reasoning_quality_score", n_resamples=10000, random_state=42)
            h01_results["reasoning_quality_score"][scenario_name] = h01_reasoning
            print(f"  Status: ✓ EXECUTED")
            for c in h01_reasoning.per_category:
                if c.n >= 4:
                    print(f"    {c.category:25s}: IQM={c.iqm:5.2f} [{c.ci_lower:5.2f}, {c.ci_upper:5.2f}]  n={c.n}")
                else:
                    print(f"    {c.category:25s}: ✗ SKIPPED (n={c.n} < 4)")
        except Exception as e:
            print(f"  Status: ✗ ERROR — {str(e)[:60]}")
    else:
        print(f"  Status: ✗ SKIPPED (no samples)")
    
    # ========== H01: hallucination_score (no filter) ==========
    print("\n[H01] hallucination_score (no filter):")
    print("-" * 90)
    
    hallucination_data = build_h01_data_for_scenario(scenario_data, "hallucination_score")
    total_samples = sum(len(v) for v in hallucination_data.values() for v in v.values() if isinstance(v, list))
    print(f"  Available samples: {total_samples}")
    
    if total_samples > 0:
        try:
            h01_hallucination = run_confidence_interval_test(hallucination_data, metric_name="hallucination_score", n_resamples=10000, random_state=42)
            h01_results["hallucination_score"][scenario_name] = h01_hallucination
            print(f"  Status: ✓ EXECUTED")
            for c in h01_hallucination.per_category:
                if c.n >= 4:
                    print(f"    {c.category:25s}: IQM={c.iqm:5.2f} [{c.ci_lower:5.2f}, {c.ci_upper:5.2f}]  n={c.n}")
                else:
                    print(f"    {c.category:25s}: ✗ SKIPPED (n={c.n} < 4)")
        except Exception as e:
            print(f"  Status: ✗ ERROR — {str(e)[:60]}")
    else:
        print(f"  Status: ✗ SKIPPED (no samples)")

print("\n✓ H01 testing complete")

H01: CONFIDENCE INTERVALS TEST — ALL SCENARIOS


SCENARIO 1: ALL POPULATED

[H01] time_to_detect (filtered by fault_detected == 1):
------------------------------------------------------------------------------------------
  Eligible samples (after filtering): 210


  Status: ✓ EXECUTED
    application_fault        : IQM=   31.3s [   25.2,    37.3]  n=60
    network_fault            : IQM=   31.7s [   26.6,    36.7]  n=90
    resource_fault           : IQM=   26.1s [   20.0,    32.5]  n=60

[H01] time_to_mitigate (filtered by fault_mitigated == 1):
------------------------------------------------------------------------------------------
  Eligible samples (after filtering): 210
  Status: ✓ EXECUTED
    application_fault        : IQM=   63.0s [   51.1,    75.0]  n=60
    network_fault            : IQM=   63.2s [   54.8,    72.0]  n=90
    resource_fault           : IQM=   69.9s [   58.9,    80.2]  n=60

[H01] reasoning_quality_score (no filter):
------------------------------------------------------------------------------------------
  Available samples: 210
  Status: ✓ EXECUTED
    application_fault        : IQM= 0.78 [ 0.72,  0.82]  n=60
    network_fault            : IQM= 0.72 [ 0.68,  0.76]  n=90
    resource_fault           : IQM= 0.74 [ 0.6

---

## Section 4: Run H02 (Success Rate Estimation) on All Scenarios

**Hypothesis 2**: Wilson Score Interval provides reliable confidence bounds on success rates.

**Success Indicators**:
- fault_detection_success_rate: count runs where `fault_detected == 1`
- fault_mitigation_success_rate: count runs where `fault_mitigated == 1`
- rai_compliance_rate: count runs where `rai_check_status == "Passed"`
- security_compliance_rate: count runs where `security_compliance_status == "Compliant"`

**Certified Floor**: Lower bound of Wilson CI (conservative estimate of true success rate)

In [8]:
def build_h02_counts_for_scenario(scenario_data, success_field, success_value, section="quantitative"):
    """
    Build success/trial counts per sub-fault for H02 testing.
    
    Implements the same logic as hypothesis_framework.scripts.utils.build_subfault_counts()
    
    Args:
        scenario_data: {category: {fault_name: [runs]}}
        success_field: Field name to check (e.g., 'fault_detected', 'rai_check_status')
        success_value: Value indicating success (e.g., 1 or "Passed")
        section: Which section to extract from ('quantitative' or 'qualitative')
    
    Returns: {category: {fault_name: (successes, trials)}}
    """
    result = {}
    
    for category, faults in scenario_data.items():
        result[category] = {}
        
        for fault_name, runs in faults.items():
            successes = 0
            trials = 0
            
            for run in runs:
                if section == "quantitative":
                    field_dict = run.get("quantitative", {})
                else:
                    field_dict = run.get("qualitative", {})
                
                val = field_dict.get(success_field)
                trials += 1
                
                if val == success_value:
                    successes += 1
            
            result[category][fault_name] = (successes, trials)
    
    return result

# Test H02 on all scenarios
print("\n" + "=" * 90)
print("H02: SUCCESS RATE ESTIMATION TEST — ALL SCENARIOS")
print("=" * 90 + "\n")

h02_results = {
    "fault_detection_success_rate": {},
    "fault_mitigation_success_rate": {},
    "rai_compliance_rate": {},
    "security_compliance_rate": {},
}

for scenario_name, scenario_data in scenarios_dict.items():
    print(f"\n{'='*90}")
    print(f"{scenario_name.upper()}")
    print(f"{'='*90}")
    
    h02_results["fault_detection_success_rate"][scenario_name] = {}
    h02_results["fault_mitigation_success_rate"][scenario_name] = {}
    h02_results["rai_compliance_rate"][scenario_name] = {}
    h02_results["security_compliance_rate"][scenario_name] = {}
    
    # ========== H02: fault_detection_success_rate ==========
    print("\n[H02] fault_detection_success_rate (success_field=fault_detected, success_value=1):")
    print("-" * 90)
    
    detection_counts = build_h02_counts_for_scenario(scenario_data, "fault_detected", 1, "quantitative")
    
    if detection_counts and any(v for v in detection_counts.values()):
        try:
            h02_detection = run_success_rate_test(detection_counts, metric_name="fault_detection_success_rate")
            h02_results["fault_detection_success_rate"][scenario_name] = h02_detection
            print(f"  Status: ✓ EXECUTED")
            for c in h02_detection.per_category:
                print(f"    {c.category:25s}: {c.successes}/{c.trials} = {c.rate:6.1%}  Wilson: [{c.wilson_lower:.3f}, {c.wilson_upper:.3f}]  Floor: {c.certified_floor:.1%}")
        except Exception as e:
            print(f"  Status: ✗ ERROR — {str(e)[:60]}")
    else:
        print(f"  Status: ✗ SKIPPED (no counts)")
    
    # ========== H02: fault_mitigation_success_rate ==========
    print("\n[H02] fault_mitigation_success_rate (success_field=fault_mitigated, success_value=1):")
    print("-" * 90)
    
    mitigation_counts = build_h02_counts_for_scenario(scenario_data, "fault_mitigated", 1, "quantitative")
    
    if mitigation_counts and any(v for v in mitigation_counts.values()):
        try:
            h02_mitigation = run_success_rate_test(mitigation_counts, metric_name="fault_mitigation_success_rate")
            h02_results["fault_mitigation_success_rate"][scenario_name] = h02_mitigation
            print(f"  Status: ✓ EXECUTED")
            for c in h02_mitigation.per_category:
                print(f"    {c.category:25s}: {c.successes}/{c.trials} = {c.rate:6.1%}  Wilson: [{c.wilson_lower:.3f}, {c.wilson_upper:.3f}]  Floor: {c.certified_floor:.1%}")
        except Exception as e:
            print(f"  Status: ✗ ERROR — {str(e)[:60]}")
    else:
        print(f"  Status: ✗ SKIPPED (no counts)")
    
    # ========== H02: rai_compliance_rate ==========
    print("\n[H02] rai_compliance_rate (success_field=rai_check_status, success_value='Passed'):")
    print("-" * 90)
    
    rai_counts = build_h02_counts_for_scenario(scenario_data, "rai_check_status", "Passed", "qualitative")
    
    if rai_counts and any(v for v in rai_counts.values()):
        try:
            h02_rai = run_success_rate_test(rai_counts, metric_name="rai_compliance_rate")
            h02_results["rai_compliance_rate"][scenario_name] = h02_rai
            print(f"  Status: ✓ EXECUTED")
            for c in h02_rai.per_category:
                print(f"    {c.category:25s}: {c.successes}/{c.trials} = {c.rate:6.1%}  Wilson: [{c.wilson_lower:.3f}, {c.wilson_upper:.3f}]  Floor: {c.certified_floor:.1%}")
        except Exception as e:
            print(f"  Status: ✗ ERROR — {str(e)[:60]}")
    else:
        print(f"  Status: ✗ SKIPPED (no counts)")
    
    # ========== H02: security_compliance_rate ==========
    print("\n[H02] security_compliance_rate (success_field=security_compliance_status, success_value='Compliant'):")
    print("-" * 90)
    
    security_counts = build_h02_counts_for_scenario(scenario_data, "security_compliance_status", "Compliant", "qualitative")
    
    if security_counts and any(v for v in security_counts.values()):
        try:
            h02_security = run_success_rate_test(security_counts, metric_name="security_compliance_rate")
            h02_results["security_compliance_rate"][scenario_name] = h02_security
            print(f"  Status: ✓ EXECUTED")
            for c in h02_security.per_category:
                print(f"    {c.category:25s}: {c.successes}/{c.trials} = {c.rate:6.1%}  Wilson: [{c.wilson_lower:.3f}, {c.wilson_upper:.3f}]  Floor: {c.certified_floor:.1%}")
        except Exception as e:
            print(f"  Status: ✗ ERROR — {str(e)[:60]}")
    else:
        print(f"  Status: ✗ SKIPPED (no counts)")

print("\n✓ H02 testing complete")


H02: SUCCESS RATE ESTIMATION TEST — ALL SCENARIOS


SCENARIO 1: ALL POPULATED

[H02] fault_detection_success_rate (success_field=fault_detected, success_value=1):
------------------------------------------------------------------------------------------
  Status: ✓ EXECUTED
    application_fault        : 60/60 = 100.0%  Wilson: [0.940, 1.000]  Floor: 94.0%
    network_fault            : 90/90 = 100.0%  Wilson: [0.959, 1.000]  Floor: 95.9%
    resource_fault           : 60/60 = 100.0%  Wilson: [0.940, 1.000]  Floor: 94.0%

[H02] fault_mitigation_success_rate (success_field=fault_mitigated, success_value=1):
------------------------------------------------------------------------------------------
  Status: ✓ EXECUTED
    application_fault        : 60/60 = 100.0%  Wilson: [0.940, 1.000]  Floor: 94.0%
    network_fault            : 90/90 = 100.0%  Wilson: [0.959, 1.000]  Floor: 95.9%
    resource_fault           : 60/60 = 100.0%  Wilson: [0.940, 1.000]  Floor: 94.0%

[H02] rai_compliance

---

## Section 5: Comparison Summary Across All 5 Scenarios

Create consolidated tables and visualizations comparing H01 CI widths and H02 certified floors across scenarios.

In [9]:
print("=" * 100)
print("SUMMARY: H01 CI WIDTHS AND VARIANCE ACROSS SCENARIOS")
print("=" * 100 + "\n")

# Summarize H01 results across scenarios
for metric in ["time_to_detect", "time_to_mitigate", "reasoning_quality_score", "hallucination_score"]:
    print(f"\n{metric}:")
    print("-" * 100)
    
    print(f"{'Scenario':<30} | {'Application':<20} | {'Network':<20} | {'Resource':<20}")
    print("-" * 100)
    
    for scenario_name in scenarios_dict.keys():
        if scenario_name in h01_results[metric]:
            h01_result = h01_results[metric][scenario_name]
            
            # Check if result is not an empty dict
            if h01_result and not isinstance(h01_result, dict):
                # Get CI width for each category
                app_width = "—"
                net_width = "—"
                res_width = "—"
                
                for c in h01_result.per_category:
                    if c.n >= 4:
                        width_str = f"{c.ci_width:.1f}s" if "time" in metric else f"{c.ci_width:.3f}"
                        if c.category == "application_fault":
                            app_width = width_str
                        elif c.category == "network_fault":
                            net_width = width_str
                        elif c.category == "resource_fault":
                            res_width = width_str
                    else:
                        width_str = "SKIP"
                        if c.category == "application_fault":
                            app_width = width_str
                        elif c.category == "network_fault":
                            net_width = width_str
                        elif c.category == "resource_fault":
                            res_width = width_str
                
                print(f"{scenario_name:<30} | {app_width:<20} | {net_width:<20} | {res_width:<20}")
            else:
                print(f"{scenario_name:<30} | SKIPPED (no result)")
        else:
            print(f"{scenario_name:<30} | NO RESULTS")

# Summarize H02 results across scenarios
print("\n\n" + "=" * 100)
print("SUMMARY: H02 CERTIFIED FLOORS (WILSON CI LOWER BOUND) ACROSS SCENARIOS")
print("=" * 100 + "\n")

for metric in ["fault_detection_success_rate", "fault_mitigation_success_rate", "rai_compliance_rate", "security_compliance_rate"]:
    print(f"\n{metric}:")
    print("-" * 100)
    
    print(f"{'Scenario':<30} | {'Application':<20} | {'Network':<20} | {'Resource':<20}")
    print("-" * 100)
    
    for scenario_name in scenarios_dict.keys():
        if scenario_name in h02_results[metric]:
            h02_result = h02_results[metric][scenario_name]
            
            # Check if result is not an empty dict
            if h02_result and not isinstance(h02_result, dict):
                # Get certified floor for each category
                app_floor = "—"
                net_floor = "—"
                res_floor = "—"
                
                for c in h02_result.per_category:
                    floor_str = f"{c.certified_floor:.1%}"
                    if c.category == "application_fault":
                        app_floor = floor_str
                    elif c.category == "network_fault":
                        net_floor = floor_str
                    elif c.category == "resource_fault":
                        res_floor = floor_str
                
                print(f"{scenario_name:<30} | {app_floor:<20} | {net_floor:<20} | {res_floor:<20}")
            else:
                print(f"{scenario_name:<30} | SKIPPED (no result)")
        else:
            print(f"{scenario_name:<30} | NO RESULTS")

# Final insights
print("\n\n" + "=" * 100)
print("KEY INSIGHTS")
print("=" * 100 + "\n")

print("✓ Scenario 1 (All Populated):")
print("  - Both H01 and H02 execute successfully for all metrics")
print("  - H01 produces actionable CI bounds (narrow/moderate widths)")
print("  - H02 produces confident success rate floors\n")

print("✓ Scenario 2 (All Missing):")
print("  - H01 skips all tests (no data after filtering)")
print("  - H02 may have 0/30 successes but tests still run")
print("  - Demonstrates data quality impact on hypothesis testing\n")

print("✓ Scenario 3 (70% Missing):")
print("  - Reduced sample size (n=9) per category")
print("  - May skip some tests (n < 4 threshold)")
print("  - Shows borderline behavior at minimum sample size\n")

print("✓ Scenario 4 (Low Variance):")
print("  - H01 produces very tight CI bounds (tight distribution)")
print("  - High confidence in narrow estimate")
print("  - H02 produces 100% success rates (all passed/compliant)\n")

print("✓ Scenario 5 (Mixed with Outliers):")
print("  - H01 produces wider CI bounds (outliers increase variance)")
print("  - Demonstrates sensitivity to extreme values")
print("  - H02 shows lower certified floors (some failures)\n")

print("✓ Scenario 6 (All Identical):")
print("  - H01 produces ZERO CI width (all values identical = no variance)")
print("  - IQM equals the single repeated value exactly")
print("  - Bootstrap resampling yields identical results every iteration")
print("  - H02 produces 100% success rates (all metrics identical)")
print("  - Demonstrates mathematical boundary case: CI = [value, value]\n")

print("✓ Filter Behavior:")
print("  - TTD filtered by fault_detected=1 reduces samples")
print("  - TTM filtered by fault_mitigated=1 may reduce samples more")
print("  - Filters are correctly applied before H01 computation")
print("  - Reasoning and hallucination have no filter (all data eligible)\n")

print("\n✓ Comparison complete")

SUMMARY: H01 CI WIDTHS AND VARIANCE ACROSS SCENARIOS


time_to_detect:
----------------------------------------------------------------------------------------------------
Scenario                       | Application          | Network              | Resource            
----------------------------------------------------------------------------------------------------
Scenario 1: All Populated      | 12.0s                | 10.1s                | 12.5s               
Scenario 2: All Missing        | SKIPPED (no result)
Scenario 3: 70% Missing        | 18.3s                | 15.5s                | 27.6s               
Scenario 4: Low Variance       | 0.0s                 | 0.0s                 | 0.0s                
Scenario 5: Mixed with Outliers | 65.7s                | 24.3s                | 44.2s               
Scenario 6: All Identical      | 0.0s                 | 0.0s                 | 0.0s                
Scenario 7: One Sample Per Fault | —                    | —              

In [12]:

print("\n\n" + "=" * 120)
print("SCENARIO 3: DETECTION & MITIGATION SUMMARY ACROSS ALL CATEGORIES")
print("=" * 120)

scenario_3_data = scenarios_dict["Scenario 3: 70% Missing"]

# Process each category
for category in sorted(scenario_3_data.keys()):
    faults = scenario_3_data[category]
    
    print(f"\n\n{category.upper()}")
    print("-" * 120)
    
    category_total_detected = 0
    category_total_mitigated = 0
    category_total_runs = 0
    
    # Create detailed breakdown per sub-fault
    sub_fault_data = []
    
    for fault_name in sorted(faults.keys()):
        runs = faults[fault_name]
        
        detected_count = 0
        mitigated_count = 0
        
        for run in runs:
            quantitative = run["quantitative"]
            detected_count += quantitative.get("fault_detected", 0)
            mitigated_count += quantitative.get("fault_mitigated", 0)
        
        total_runs = len(runs)
        detected_rate = detected_count / total_runs * 100 if total_runs > 0 else 0
        mitigated_rate = mitigated_count / total_runs * 100 if total_runs > 0 else 0
        
        sub_fault_data.append({
            "fault_name": fault_name,
            "detected": detected_count,
            "mitigated": mitigated_count,
            "total": total_runs,
            "detected_rate": detected_rate,
            "mitigated_rate": mitigated_rate,
        })
        
        category_total_detected += detected_count
        category_total_mitigated += mitigated_count
        category_total_runs += total_runs
    
    # Print header
    print(f"{'Sub-Fault Name':<30} {'Detected':<15} {'Mitigated':<15} {'Total Runs':<12} {'Detection %':<15} {'Mitigation %':<15}")
    print("-" * 120)
    
    # Print each sub-fault
    for data in sub_fault_data:
        print(f"{data['fault_name']:<30} {data['detected']:<15} {data['mitigated']:<15} {data['total']:<12} {data['detected_rate']:>6.1f}% {data['mitigated_rate']:>14.1f}%")
    
    # Print category totals
    print("-" * 120)
    category_detected_rate = category_total_detected / category_total_runs * 100 if category_total_runs > 0 else 0
    category_mitigated_rate = category_total_mitigated / category_total_runs * 100 if category_total_runs > 0 else 0
    print(f"{'CATEGORY TOTAL':<30} {category_total_detected:<15} {category_total_mitigated:<15} {category_total_runs:<12} {category_detected_rate:>6.1f}% {category_mitigated_rate:>14.1f}%")
    print()

# Print overall summary
print("\n" + "=" * 120)
print("SCENARIO 3: OVERALL SUMMARY (ALL CATEGORIES COMBINED)")
print("=" * 120)

overall_detected = 0
overall_mitigated = 0
overall_runs = 0

for category, faults in scenario_3_data.items():
    for fault_name, runs in faults.items():
        for run in runs:
            quantitative = run["quantitative"]
            overall_detected += quantitative.get("fault_detected", 0)
            overall_mitigated += quantitative.get("fault_mitigated", 0)
            overall_runs += 1

overall_detected_rate = overall_detected / overall_runs * 100 if overall_runs > 0 else 0
overall_mitigated_rate = overall_mitigated / overall_runs * 100 if overall_runs > 0 else 0

print(f"\nTotal Runs Across All Categories:        {overall_runs}")
print(f"Runs with fault_detected = 1:            {overall_detected:>3} ({overall_detected_rate:>6.1f}%)")
print(f"Runs with fault_mitigated = 1:           {overall_mitigated:>3} ({overall_mitigated_rate:>6.1f}%)")

print(f"\nKey Insight for Scenario 3 (70% Missing):")
print(f"  • Only ~30% of runs have data (fault_detected=1 or fault_mitigated=1)")
print(f"  • The remaining ~70% have None values in time_to_detect/time_to_mitigate")
print(f"  • These missing values are treated as failures for H02 (success rate) calculations")
print(f"  • This is why certified floors are conservative (low) for Scenario 3")

print("\n✓ Scenario 3 detection/mitigation summary complete")




SCENARIO 3: DETECTION & MITIGATION SUMMARY ACROSS ALL CATEGORIES


APPLICATION_FAULT
------------------------------------------------------------------------------------------------------------------------
Sub-Fault Name                 Detected        Mitigated       Total Runs   Detection %     Mitigation %   
------------------------------------------------------------------------------------------------------------------------
pod-delete                     12              12              30             40.0%           40.0%
pod-restart                    13              13              30             43.3%           43.3%
------------------------------------------------------------------------------------------------------------------------
CATEGORY TOTAL                 25              25              60             41.7%           41.7%



NETWORK_FAULT
------------------------------------------------------------------------------------------------------------------------
Sub-